# JSON in, answers out

Which regions earn, who buys most, and what is still open? The steps of `demos/02_orders.rn`, run from this directory, so the data is `../data/orders.json`. Money stays in integer cents.

In [1]:
fn dollars(cents) { format!("{}.{:02}", cents / 100, cents % 100) }

// add `cents` to the [name, total] row for `name`, making the row if new
fn add(rows, name, cents) {
    for row in rows { if row[0] == name { row[1] = row[1] + cents; return; } }
    rows.push([name, cents]);
}

// largest total first; equal totals by name
fn by_total(a, b) { if a[1] != b[1] { b[1].cmp(a[1]) } else { a[0].cmp(b[0]) } }

Read the orders. The cell's value is how many there are.

In [2]:
let path = "../data/orders.json";
let orders = json::parse(fs::read(path)?)?;
orders.len()

12

Revenue counts shipped orders only; open and cancelled orders add nothing. The cell's value is the number of open orders.

In [3]:
let regions = [];
let customers = [];
let open = 0;
for o in orders {
    let cents = if o.status == "shipped" { o.qty * o.unit_cents } else { 0 };
    if o.status == "open" { open += 1; }
    add(regions, o.region, cents);
    add(customers, o.customer, cents);
}
open

4

Sort, then answer.

In [4]:
regions.sort_by(by_total);
customers.sort_by(by_total);
println!("{} orders read from {}", orders.len(), path);
println!("Shipped revenue by region:");
for r in regions { println!("  {:<6} {:>8}", r[0], dollars(r[1])); }
println!("Largest customer: {} ({}), ahead of {} ({})",
    customers[0][0], dollars(customers[0][1]), customers[1][0], dollars(customers[1][1]));
println!("Still open: {} of {} orders ({:.1}%)", open, orders.len(), open as f64 * 100.0 / orders.len() as f64);

12 orders read from ../data/orders.json
Shipped revenue by region:
  north    245.00
  east     144.97
  south     99.95
  west       0.00
Largest customer: Acme (134.99), ahead of Birch (134.98)
Still open: 4 of 12 orders (33.3%)
